In [ ]:
from IPython.display import Image

Image(filename='UC-NEGRO-01.jpg', width=200)

# IIC-3670 NLP UC

- Versiones de librerías, python 3.10

- pandas 2.2
- torch 2.7
- transformers 4.41
- peft 0.6

Con LoRA, un mismo modelo base sirve para varias tareas: cada tarea guarda solo su adaptador, y se cambia de tarea cambiando de adaptador. La pregunta es cuánto se gana frente a entrenar solo un cabezal sobre el encoder congelado, y qué pasa si se usa el adaptador equivocado.

Las tareas son dos: sentimiento en críticas de cine (SST-2, dos clases) y categoría de un titular de noticias (cuatro clases).

In [ ]:
import os
os.environ['USE_TF'] = '0'

import random
import time
import datasets
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

transformers.logging.set_verbosity_error()
datasets.logging.set_verbosity_error()
datasets.disable_progress_bar()
random.seed(0)

sst = load_dataset('glue', 'sst2')
tr = sst['train'].shuffle(seed=0).select(range(2000))
SST2 = {'train': (tr['sentence'], tr['label']),
        'test': (sst['validation']['sentence'], sst['validation']['label'])}

CATEGORIAS = ['POLITICS', 'ENTERTAINMENT', 'SPORTS', 'BUSINESS']
nw = load_dataset('heegyu/news-category-dataset')['train'].shuffle(seed=0)
por_cat = {c: [] for c in CATEGORIAS}
for h, c in zip(nw['headline'], nw['category']):
    if c in por_cat and len(por_cat[c]) < 700:
        por_cat[c].append(h)
entrenamiento = [(h, i) for i, c in enumerate(CATEGORIAS) for h in por_cat[c][:500]]
prueba = [(h, i) for i, c in enumerate(CATEGORIAS) for h in por_cat[c][500:]]
random.shuffle(entrenamiento)
NOTICIAS = {'train': tuple(map(list, zip(*entrenamiento))), 'test': tuple(map(list, zip(*prueba)))}
TAREAS = {'sst2': (SST2, 2), 'noticias': (NOTICIAS, 4)}

for nombre, (datos, c) in TAREAS.items():
    print(f"{nombre}: {len(datos['train'][0])} de entrenamiento, {len(datos['test'][0])} de prueba, {c} clases")
    print('   ', datos['train'][0][0], '->', datos['train'][1][0])

El encoder es **distilroberta-base** (82 millones de parámetros). La representación de un texto es el promedio de los vectores de salida de sus piezas, sin contar el relleno. Las funciones de abajo arman los lotes, calculan esa representación y evalúan un encoder con un cabezal lineal; úselas tal cual.

In [ ]:
MODELO = 'distilroberta-base'
tok = AutoTokenizer.from_pretrained(MODELO)


def lotes(X, Y, b=16, barajar=False):
    idx = list(range(len(X)))
    if barajar:
        random.shuffle(idx)
    for i in range(0, len(idx), b):
        j = idx[i:i + b]
        enc = tok([X[k] for k in j], padding=True, truncation=True, max_length=64, return_tensors='pt')
        yield enc, torch.tensor([int(Y[k]) for k in j])


def representar(encoder, enc):
    h = encoder(**enc).last_hidden_state
    m = enc['attention_mask'].unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1)


@torch.no_grad()
def evaluar(encoder, cabezal, datos):
    encoder.eval()
    aciertos = sum((cabezal(representar(encoder, enc)).argmax(-1) == y).sum().item()
                   for enc, y in lotes(*datos['test'], b=64))
    return aciertos / len(datos['test'][1])

## Actividad en clase

1. **Solo el cabezal.** Congele distilroberta-base y entrene, para cada tarea, un cabezal lineal sobre sus representaciones (conviene calcularlas una vez y entrenar el cabezal 50 épocas sobre ellas). Reporte la exactitud en prueba y los parámetros entrenados.
2. **Ajuste completo.** Como referencia, ajuste todo el encoder más el cabezal durante una época, con tasa de aprendizaje 3e-5.
3. **Dos adaptadores.** Con `peft`, envuelva **un solo** encoder con un adaptador LoRA llamado `sst2` (r = 8, α = 16, sobre `query` y `value`) y agréguele un segundo, `noticias`, con `add_adapter`. Entrene cada adaptador con su propio cabezal durante una época (tasa 1e-3), activándolo con `set_adapter`. Reporte exactitud, parámetros entrenados y el tamaño en disco de cada adaptador (`save_pretrained`).
4. **Cambiar de adaptador.** Evalúe las dos tareas con cada adaptador activo, y también con los adaptadores desactivados (`disable_adapter`). Arme una tabla de 3 filas por 2 tareas.
5. Compare los tres métodos. ¿Cuánto cuesta cada tarea nueva en cada uno?

- Cuando termine, me avisa para entregarle una **L (logrado)**.
- Recuerde que las L otorgan un bono en la nota final de la asignatura.

***Tiene hasta el final de la clase.***